In [1]:
# TEMPORAL PART1 — CELL 01
# RAW TIMESTAMP SOURCE DISCOVERY & READINESS
#
# READ ONLY | NO MODEL TRAINING | NO DATA MODIFICATION
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

MODEL_PATH = (
    ROOT / "04_Modeling" / "outputs"
    / "leakage_grouping"
    / "modeling_labeled_348_grouped.csv"
)

RAW_DIR = ROOT / "01_Raw Data" / "outputs" / "raw_csv"

EXPECTED_MODEL_SHA = (
    "cfa7df27079ef47bdb1b0cb526bc017aa1c8fe18dc4d91fef870a3441e2e9fb7"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

assert MODEL_PATH.is_file(), "Dataset Modeling tidak ditemukan"
assert RAW_DIR.is_dir(), "Folder raw_csv tidak ditemukan"

assert sha256_file(MODEL_PATH) == EXPECTED_MODEL_SHA, (
    "Dataset Modeling berbeda dari versi terkunci"
)

model = pd.read_csv(
    MODEL_PATH,
    encoding="utf-8-sig",
    dtype={"id": "string"},
    keep_default_na=False
)

assert len(model) == 348
assert model["id"].is_unique

assert {"id", "case", "sentiment_label"}.issubset(model.columns)

raw_files = sorted(RAW_DIR.rglob("*.csv"))

assert raw_files, "Tidak ada raw CSV ditemukan"

target_ids = set(model["id"].astype(str))

matches = []
files_with_createdat = 0
files_without_createdat = []
read_errors = []
raw_rows_scanned = 0

for path in raw_files:
    try:
        header = pd.read_csv(
            path,
            nrows=0,
            encoding="utf-8-sig"
        )

        if not {"id", "createdAt"}.issubset(header.columns):
            files_without_createdat.append(
                str(path.relative_to(ROOT))
            )
            continue

        files_with_createdat += 1

        raw = pd.read_csv(
            path,
            usecols=["id", "createdAt"],
            encoding="utf-8-sig",
            dtype={"id": "string", "createdAt": "string"},
            keep_default_na=False,
            on_bad_lines="error"
        )

        raw_rows_scanned += len(raw)

        subset = raw.loc[
            raw["id"].isin(target_ids)
        ].copy()

        if not subset.empty:
            subset["source_file"] = str(
                path.relative_to(ROOT)
            ).replace("\\", "/")

            matches.append(subset)

    except Exception as error:
        read_errors.append({
            "file": str(path.relative_to(ROOT)),
            "error": str(error)[:300]
        })

if matches:
    matched = pd.concat(matches, ignore_index=True)
else:
    matched = pd.DataFrame(
        columns=["id", "createdAt", "source_file"]
    )

matched["createdAt"] = (
    matched["createdAt"]
    .astype("string")
    .str.strip()
)

matched["parsed_utc"] = pd.to_datetime(
    matched["createdAt"],
    utc=True,
    errors="coerce"
)

matched["timestamp_valid"] = matched["parsed_utc"].notna()

# Count different VALID timestamps for each tweet ID.
valid_matches = matched.loc[matched["timestamp_valid"]]

timestamp_conflicts = (
    valid_matches.groupby("id")["parsed_utc"]
    .nunique()
)

conflict_ids = set(
    timestamp_conflicts[timestamp_conflicts > 1].index
)

valid_ids = set(valid_matches["id"].astype(str))
matched_ids = set(matched["id"].astype(str))

missing_ids = target_ids - matched_ids
matched_without_valid_time = matched_ids - valid_ids

print("=" * 72)
print("05_TEMPORAL — PART1 TIMESTAMP READINESS")
print("=" * 72)

print("Locked Modeling rows        :", len(model))
print("Unique Modeling IDs         :", model["id"].nunique())
print("Raw CSV files found         :", len(raw_files))
print("Files with createdAt        :", files_with_createdat)
print("Raw rows scanned            :", raw_rows_scanned)
print("Raw matching records        :", len(matched))
print("Unique matched IDs          :", len(matched_ids))
print("Unique valid timestamp IDs  :", len(valid_ids))
print("Missing raw IDs             :", len(missing_ids))
print("Matched IDs no valid time   :", len(matched_without_valid_time))
print("Conflicting timestamp IDs   :", len(conflict_ids))
print("Raw file read errors        :", len(read_errors))

print("\nMODEL DISTRIBUTION BY CASE")
display(
    model.groupby("case")
    .agg(
        tweet_count=("id", "nunique")
    )
    .reset_index()
)

print("\nTIMESTAMP COVERAGE BY CASE")
coverage = model[["id", "case"]].copy()

coverage["found_raw"] = coverage["id"].isin(matched_ids)
coverage["valid_timestamp"] = coverage["id"].isin(valid_ids)
coverage["timestamp_conflict"] = coverage["id"].isin(conflict_ids)

display(
    coverage.groupby("case")
    .agg(
        tweets=("id", "count"),
        found_raw=("found_raw", "sum"),
        valid_timestamp=("valid_timestamp", "sum"),
        conflicting_timestamp=("timestamp_conflict", "sum")
    )
    .reset_index()
)

if not valid_matches.empty:
    print("\nEARLIEST / LATEST RAW TIMESTAMPS (UTC)")
    print("Earliest:", valid_matches["parsed_utc"].min())
    print("Latest  :", valid_matches["parsed_utc"].max())

    first_valid = (
        valid_matches.sort_values(["id", "parsed_utc"])
        .drop_duplicates("id")
        [["id", "parsed_utc"]]
    )

    date_preview = model[["id", "case"]].merge(
        first_valid,
        on="id",
        how="left",
        validate="one_to_one"
    )

    date_preview["date_wib"] = (
        date_preview["parsed_utc"]
        .dt.tz_convert("Asia/Jakarta")
        .dt.date
    )

    print("\nDATE COVERAGE BY CASE (WIB)")
    display(
        date_preview.groupby("case")
        .agg(
            earliest=("date_wib", "min"),
            latest=("date_wib", "max"),
            rows_with_dates=("parsed_utc", "count")
        )
        .reset_index()
    )

if read_errors:
    print("\nRAW FILE READ ERRORS (FIRST 10)")
    display(pd.DataFrame(read_errors).head(10))

if files_without_createdat:
    print("\nFILES WITHOUT TIMESTAMP COLUMN (FIRST 10)")
    print("\n".join(files_without_createdat[:10]))

print("\nFINAL PREFLIGHT DECISION")

if (
    len(valid_ids) == 348
    and len(conflict_ids) == 0
    and len(read_errors) == 0
):
    print("TEMPORAL TIMESTAMP PREFLIGHT: PASS")
else:
    print("TEMPORAL TIMESTAMP PREFLIGHT: REVIEW REQUIRED")

print("READ-ONLY SOURCE SCAN COMPLETE")
print("=" * 72)


05_TEMPORAL — PART1 TIMESTAMP READINESS
Locked Modeling rows        : 348
Unique Modeling IDs         : 348
Raw CSV files found         : 65
Files with createdAt        : 57
Raw rows scanned            : 1388
Raw matching records        : 831
Unique matched IDs          : 348
Unique valid timestamp IDs  : 348
Missing raw IDs             : 0
Matched IDs no valid time   : 0
Conflicting timestamp IDs   : 0
Raw file read errors        : 0

MODEL DISTRIBUTION BY CASE


C:\Users\Yusak SR\AppData\Local\Temp\ipykernel_1372\4237529174.py:124: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  matched["parsed_utc"] = pd.to_datetime(


,case,tweet_count
0,lombok_2018,101
1,palu_2018,105
2,sumatra_2025,142



TIMESTAMP COVERAGE BY CASE


,case,tweets,found_raw,valid_timestamp,conflicting_timestamp
0,lombok_2018,101,101,101,0
1,palu_2018,105,105,105,0
2,sumatra_2025,142,142,142,0



EARLIEST / LATEST RAW TIMESTAMPS (UTC)
Earliest: 2018-08-06 07:40:02+00:00
Latest  : 2025-12-04 14:43:34+00:00

DATE COVERAGE BY CASE (WIB)


,case,earliest,latest,rows_with_dates
0,lombok_2018,2018-08-06,2018-08-14,101
1,palu_2018,2018-09-29,2018-10-03,105
2,sumatra_2025,2025-11-29,2025-12-04,142



FILES WITHOUT TIMESTAMP COLUMN (FIRST 10)
01_Raw Data\outputs\raw_csv\lombok_2018\lombok_2018_539338a38118f5470d1b.csv
01_Raw Data\outputs\raw_csv\lombok_2018\lombok_2018_7085e181db6c8ff0911a.csv
01_Raw Data\outputs\raw_csv\lombok_2018\lombok_2018_971bf25151d9f4419d91.csv
01_Raw Data\outputs\raw_csv\palu_2018\palu_2018_0ac1c9ee684c47c748ff.csv
01_Raw Data\outputs\raw_csv\palu_2018\palu_2018_b4187151cd0d81be2864.csv
01_Raw Data\outputs\raw_csv\sumatra_2025\sumatra_2025_04ed36216ec543478f4a.csv
01_Raw Data\outputs\raw_csv\sumatra_2025\sumatra_2025_4f9a9ad2f4ca2fcd88d6.csv
01_Raw Data\outputs\raw_csv\sumatra_2025\sumatra_2025_a2a75c0483636c3341eb.csv

FINAL PREFLIGHT DECISION
TEMPORAL TIMESTAMP PREFLIGHT: PASS
READ-ONLY SOURCE SCAN COMPLETE


In [2]:
# TEMPORAL PART2 — CELL 02
# TIMESTAMP CANONICALIZATION & IMMUTABLE DATASET LOCK
#
# READ ONLY SOURCES | ZERO MODEL TRAINING
# STRICT TIMESTAMP CONFLICT CHECK
# SHA256 PROVENANCE | EXCLUSIVE OUTPUT CREATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

MODEL_PATH = (
    ROOT / "04_Modeling" / "outputs"
    / "leakage_grouping"
    / "modeling_labeled_348_grouped.csv"
)

RAW_DIR = ROOT / "01_Raw Data" / "outputs" / "raw_csv"

OUT_DIR = (
    ROOT / "05_Temporal" / "outputs"
    / "timestamp_canonicalization"
)

CANONICAL_PATH = OUT_DIR / "temporal_labeled_348_locked.csv"
PROVENANCE_PATH = OUT_DIR / "timestamp_source_provenance.csv"
COVERAGE_PATH = OUT_DIR / "timestamp_coverage_by_case.csv"
MANIFEST_PATH = OUT_DIR / "temporal_timestamp_lock_manifest.json"
AUDIT_PATH = OUT_DIR / "part2_timestamp_lock_audit.json"

OUTPUTS = [
    CANONICAL_PATH,
    PROVENANCE_PATH,
    COVERAGE_PATH,
    MANIFEST_PATH,
    AUDIT_PATH,
]

EXPECTED_MODEL_SHA = (
    "cfa7df27079ef47bdb1b0cb526bc017aa1c8fe18dc4d91fef870a3441e2e9fb7"
)

EXPECTED_CASES = {
    "lombok_2018": 101,
    "palu_2018": 105,
    "sumatra_2025": 142,
}

EXPECTED_LABELS = {
    "NETRAL": 225,
    "POSITIF": 108,
    "NEGATIF": 15,
}

TWITTER_DATE_FORMAT = "%a %b %d %H:%M:%S %z %Y"

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def relpath(path):
    return path.relative_to(ROOT).as_posix()

def write_json_temp(path, payload):
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

print("=" * 76)
print("TEMPORAL PART2 — TIMESTAMP CANONICALIZATION")
print("=" * 76)

# ------------------------------------------------------------
# A. PREFLIGHT — LOCKED INPUT AND OUTPUT COLLISION
# ------------------------------------------------------------

assert MODEL_PATH.is_file(), "Locked Modeling file missing"
assert RAW_DIR.is_dir(), "Raw CSV directory missing"

for output in OUTPUTS:
    if output.exists():
        raise FileExistsError(
            f"Output already exists; refusing overwrite: {output}"
        )

assert sha256_file(MODEL_PATH) == EXPECTED_MODEL_SHA, (
    "Locked Modeling dataset SHA256 mismatch"
)

model = pd.read_csv(
    MODEL_PATH,
    encoding="utf-8-sig",
    dtype={"id": "string"},
    keep_default_na=False,
)

assert len(model) == 348
assert model["id"].is_unique

required_model = {
    "id", "case", "sentiment_label",
    "leakage_group_id"
}
assert required_model.issubset(model.columns)

assert model["case"].value_counts().to_dict() == EXPECTED_CASES
assert model["sentiment_label"].value_counts().to_dict() == EXPECTED_LABELS

raw_files = sorted(RAW_DIR.rglob("*.csv"))
assert len(raw_files) > 0

# Hash every file before reading. All raw sources are immutable.
source_hashes = {
    relpath(MODEL_PATH): sha256_file(MODEL_PATH)
}
source_hashes.update({
    relpath(path): sha256_file(path)
    for path in raw_files
})

target_keys = set(
    zip(
        model["case"].astype(str),
        model["id"].astype(str)
    )
)

print("Locked rows:", len(model))
print("Raw CSV files:", len(raw_files))

# ------------------------------------------------------------
# B. SCAN RAW FILES AND STRICTLY PARSE TIMESTAMPS
# ------------------------------------------------------------

raw_matches = []
files_without_timestamp = []
read_errors = []
raw_rows_scanned = 0
files_with_timestamp = 0
matched_files = set()

for path in raw_files:
    source_name = relpath(path)

    try:
        header = pd.read_csv(
            path, nrows=0, encoding="utf-8-sig"
        )

        if not {"id", "createdAt"}.issubset(header.columns):
            files_without_timestamp.append(source_name)
            continue

        files_with_timestamp += 1

        raw = pd.read_csv(
            path,
            usecols=["id", "createdAt"],
            dtype={"id": "string", "createdAt": "string"},
            encoding="utf-8-sig",
            keep_default_na=False,
            on_bad_lines="error",
        )

        raw_rows_scanned += len(raw)

        # Expected path: raw_csv/<case>/<filename>.csv
        source_case = path.parent.name

        if source_case not in EXPECTED_CASES:
            raise AssertionError(
                f"Unknown source case: {source_case}"
            )

        raw["id"] = raw["id"].str.strip()
        raw["createdAt"] = raw["createdAt"].str.strip()

        subset = raw.loc[
            raw["id"].isin(model["id"])
        ].copy()

        # Enforce that each matched tweet belongs to its
        # original event/case; do not accept cross-case IDs.
        if not subset.empty:
            subset["case"] = source_case

            pairs = list(zip(
                subset["case"].astype(str),
                subset["id"].astype(str)
            ))

            if not all(key in target_keys for key in pairs):
                raise AssertionError(
                    f"Cross-case ID mismatch in {source_name}"
                )

            subset["source_file"] = source_name
            subset["source_file_sha256"] = source_hashes[source_name]
            subset["source_row_number"] = (
                subset.index.to_numpy() + 2
            )

            raw_matches.append(subset)
            matched_files.add(source_name)

    except Exception as exc:
        read_errors.append({
            "file": source_name,
            "error": str(exc)[:500]
        })

assert not read_errors, (
    f"Raw source errors detected: {read_errors[:5]}"
)

assert raw_matches, "No matching timestamp sources found"

matches = pd.concat(raw_matches, ignore_index=True)

assert matches["id"].notna().all()
assert (matches["createdAt"] != "").all(), (
    "Matching raw records contain blank timestamps"
)

# A fixed parser prevents pandas/dateutil guesswork.
# Twitter API-style: Mon Aug 13 04:49:43 +0000 2018
matches["created_at_utc"] = pd.to_datetime(
    matches["createdAt"],
    format=TWITTER_DATE_FORMAT,
    errors="coerce",
    utc=True
)

invalid_time = matches["created_at_utc"].isna()

assert not invalid_time.any(), (
    "Unparseable raw timestamps: "
    + repr(
        matches.loc[
            invalid_time,
            ["id", "createdAt", "source_file"]
        ].head(10).to_dict("records")
    )
)

# Full precision: timestamps may differ by seconds.
timestamps_per_id = (
    matches.groupby("id")["created_at_utc"]
    .nunique()
)

conflict_ids = (
    timestamps_per_id[timestamps_per_id > 1]
    .index.tolist()
)

assert not conflict_ids, (
    f"Contradictory timestamps for IDs: {conflict_ids[:10]}"
)

matched_ids = set(matches["id"].astype(str))
model_ids = set(model["id"].astype(str))

assert matched_ids == model_ids, (
    f"Missing IDs: {sorted(model_ids - matched_ids)[:10]}"
)

assert matches.groupby("id")["case"].nunique().max() == 1

print("Matching raw occurrences:", len(matches))
print("Unique matched IDs:", len(matched_ids))
print("Timestamp conflicts:", len(conflict_ids))
print("Raw read errors:", len(read_errors))

# ------------------------------------------------------------
# C. DETERMINISTIC CANONICAL TIMESTAMP SELECTION
# ------------------------------------------------------------

# Choose source deterministically only AFTER proving every
# occurrence of the same ID has exactly the same timestamp.
matches = matches.sort_values(
    ["case", "id", "source_file", "source_row_number"],
    kind="mergesort"
).reset_index(drop=True)

matches["is_canonical_source"] = (
    ~matches.duplicated(subset=["id"])
)

primary = matches.loc[
    matches["is_canonical_source"]
].copy()

assert len(primary) == 348
assert primary["id"].is_unique

source_occurrences = (
    matches.groupby("id").size().rename(
        "raw_occurrence_count"
    )
)

source_file_counts = (
    matches.groupby("id")["source_file"].nunique()
    .rename("raw_source_file_count")
)

primary = primary.set_index("id")

primary = primary.join(
    source_occurrences
).join(
    source_file_counts
)

# Preserve the 348 locked Modeling rows and their order.
canonical = model[[
    "case",
    "id",
    "sentiment_label",
    "leakage_group_id"
]].copy()

canonical = canonical.merge(
    primary.reset_index()[[
        "id", "createdAt", "created_at_utc",
        "source_file", "source_file_sha256",
        "source_row_number", "raw_occurrence_count",
        "raw_source_file_count"
    ]],
    on="id",
    how="left",
    validate="one_to_one",
    sort=False
)

assert len(canonical) == 348
assert canonical["id"].is_unique
assert canonical["created_at_utc"].notna().all()

# UTC ISO 8601, +00:00 explicitly.
canonical["created_at_utc"] = (
    canonical["created_at_utc"]
    .dt.strftime("%Y-%m-%dT%H:%M:%S%z")
)

# Convert separately from canonical UTC strings.
utc_dt = pd.to_datetime(
    canonical["created_at_utc"],
    utc=True,
    format="%Y-%m-%dT%H:%M:%S%z"
)

wib_dt = utc_dt.dt.tz_convert("Asia/Jakarta")

canonical["created_at_wib"] = (
    wib_dt.dt.strftime("%Y-%m-%dT%H:%M:%S%z")
)
canonical["date_wib"] = wib_dt.dt.strftime("%Y-%m-%d")
canonical["hour_wib"] = wib_dt.dt.hour.astype(int)

canonical = canonical.rename(columns={
    "createdAt": "raw_createdAt",
    "source_file": "canonical_source_file",
    "source_file_sha256": "canonical_source_sha256",
    "source_row_number": "canonical_source_row_number"
})

canonical = canonical[[
    "id", "case", "sentiment_label", "leakage_group_id",
    "raw_createdAt", "created_at_utc", "created_at_wib",
    "date_wib", "hour_wib",
    "canonical_source_file", "canonical_source_sha256",
    "canonical_source_row_number",
    "raw_occurrence_count", "raw_source_file_count"
]]

# Verify labels, groups and case have not been altered.
for col in [
    "id", "case", "sentiment_label", "leakage_group_id"
]:
    assert np.array_equal(
        canonical[col].astype(str).to_numpy(),
        model[col].astype(str).to_numpy()
    ), f"Locked value mismatch: {col}"

# Strictly verify the timezone offset relationship.
assert (
    wib_dt.dt.tz_convert("UTC") == utc_dt
).all()

# ------------------------------------------------------------
# D. SOURCE PROVENANCE FOR EVERY RAW OCCURRENCE
# ------------------------------------------------------------

provenance = matches[[
    "id", "case", "createdAt", "created_at_utc",
    "source_file", "source_file_sha256",
    "source_row_number", "is_canonical_source"
]].copy()

provenance["created_at_utc"] = (
    provenance["created_at_utc"]
    .dt.strftime("%Y-%m-%dT%H:%M:%S%z")
)

provenance = provenance.rename(columns={
    "createdAt": "raw_createdAt"
})

assert provenance["is_canonical_source"].sum() == 348
assert set(provenance["id"]) == model_ids

# ------------------------------------------------------------
# E. TEMPORAL COVERAGE BY CASE
# ------------------------------------------------------------

coverage = (
    canonical.groupby("case", as_index=False)
    .agg(
        tweet_count=("id", "nunique"),
        earliest_date_wib=("date_wib", "min"),
        latest_date_wib=("date_wib", "max"),
        raw_occurrences=("raw_occurrence_count", "sum")
    )
    .sort_values("case")
    .reset_index(drop=True)
)

assert dict(zip(
    coverage["case"], coverage["tweet_count"]
)) == EXPECTED_CASES

print("\nTEMPORAL COVERAGE")
print(coverage.to_string(index=False))

# ------------------------------------------------------------
# F. MANIFEST AND AUDIT
# ------------------------------------------------------------

now = datetime.now(timezone.utc).isoformat()

manifest = {
    "stage": "05_Temporal",
    "part": "2 Timestamp Canonicalization & Lock",
    "generated_at_utc": now,
    "source_commit":
        "4e29215d8a7ee1b5d98943b2edcb70f7538060aa",
    "dataset_id": "TEMPORAL_LABELED_348_V1",
    "modeling_source_sha256": EXPECTED_MODEL_SHA,
    "timestamp_source": "raw_csv.createdAt",
    "raw_timestamp_format": TWITTER_DATE_FORMAT,
    "canonical_timezone": "UTC",
    "analysis_timezone": "Asia/Jakarta",
    "canonicalization_policy": (
        "Accept only if all raw occurrences of each ID "
        "have an identical valid timestamp; choose "
        "lexicographically first source file and lowest "
        "source row as deterministic provenance."
    ),
    "modeling_rows": 348,
    "canonical_rows": len(canonical),
    "unique_ids": int(canonical["id"].nunique()),
    "matching_raw_occurrences": len(matches),
    "raw_files_scanned": len(raw_files),
    "raw_files_with_timestamp": files_with_timestamp,
    "raw_files_without_timestamp":
        files_without_timestamp,
    "matched_raw_files": sorted(matched_files),
    "raw_rows_scanned": raw_rows_scanned,
    "case_distribution": EXPECTED_CASES,
    "label_distribution": EXPECTED_LABELS,
    "timestamp_conflicts": len(conflict_ids),
    "missing_timestamp_ids": 0,
    "source_sha256": source_hashes,
    "output_sha256": {},
    "status": "TEMPORAL_TIMESTAMP_LOCK_PREPARED"
}

audit = {
    "stage": "05_Temporal",
    "part": "2 Timestamp Integrity Audit",
    "generated_at_utc": now,
    "modeling_rows_preserved": True,
    "modeling_ids_preserved": True,
    "modeling_cases_preserved": True,
    "modeling_labels_preserved": True,
    "leakage_groups_preserved": True,
    "raw_timestamp_parse_errors": 0,
    "raw_file_read_errors": len(read_errors),
    "missing_timestamp_ids": 0,
    "conflicting_timestamp_ids": 0,
    "canonical_rows": 348,
    "canonical_unique_ids": 348,
    "source_dataset_modified": False,
    "raw_files_modified": False,
    "model_training_performed": False,
    "status": "TEMPORAL_PART2_TIMESTAMP_LOCK_PASS"
}

# ------------------------------------------------------------
# G. IMMUTABLE EXPORT — NO OVERWRITE
# ------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

for output in OUTPUTS:
    if output.exists():
        raise FileExistsError(
            f"Refusing overwrite: {output}"
        )

token = uuid.uuid4().hex[:10]
tmp = {
    path: OUT_DIR / f".part2_{token}_{i}.tmp"
    for i, path in enumerate(OUTPUTS)
}
created = []

try:
    canonical.to_csv(
        tmp[CANONICAL_PATH],
        index=False, encoding="utf-8-sig"
    )
    provenance.to_csv(
        tmp[PROVENANCE_PATH],
        index=False, encoding="utf-8-sig"
    )
    coverage.to_csv(
        tmp[COVERAGE_PATH],
        index=False, encoding="utf-8-sig"
    )

    # Verify the saved CSV, not just the in-memory frame.
    check = pd.read_csv(
        tmp[CANONICAL_PATH],
        encoding="utf-8-sig",
        dtype={"id": "string"},
        keep_default_na=False
    )

    assert len(check) == 348
    assert check["id"].is_unique
    assert set(check["id"]) == model_ids

    manifest["output_sha256"] = {
        path.name: sha256_file(tmp[path])
        for path in [
            CANONICAL_PATH, PROVENANCE_PATH, COVERAGE_PATH
        ]
    }
    manifest["status"] = "TEMPORAL_TIMESTAMP_LOCK_FINAL"

    write_json_temp(tmp[MANIFEST_PATH], manifest)

    audit["output_sha256"] = {
        **manifest["output_sha256"],
        MANIFEST_PATH.name: sha256_file(tmp[MANIFEST_PATH])
    }

    write_json_temp(tmp[AUDIT_PATH], audit)

    # All source hashes must still match before writing.
    for src in [MODEL_PATH] + raw_files:
        assert sha256_file(src) == source_hashes[relpath(src)], (
            f"Source modified during execution: {src}"
        )

    for output in OUTPUTS:
        with output.open("xb") as handle:
            handle.write(tmp[output].read_bytes())
        created.append(output)

except Exception:
    for output in created:
        if output.exists():
            output.unlink()
    raise

finally:
    for temporary in tmp.values():
        if temporary.exists():
            temporary.unlink()

# ------------------------------------------------------------
# H. FINAL OUTPUT / SOURCE VERIFICATION
# ------------------------------------------------------------

for output_name, expected_sha in audit["output_sha256"].items():
    actual = sha256_file(OUT_DIR / output_name)
    assert actual == expected_sha, (
        f"Output SHA mismatch: {output_name}"
    )

for src in [MODEL_PATH] + raw_files:
    assert sha256_file(src) == source_hashes[relpath(src)]

assert json.loads(
    AUDIT_PATH.read_text(encoding="utf-8")
)["status"] == "TEMPORAL_PART2_TIMESTAMP_LOCK_PASS"

print("\n" + "=" * 76)
print("TEMPORAL PART2 — FINAL AUDIT")
print("=" * 76)
print("Canonical tweet rows       :", len(canonical))
print("Unique tweet IDs           :", canonical["id"].nunique())
print("Matching raw occurrences   :", len(provenance))
print("Canonical timestamp missing:", 0)
print("Timestamp conflicts        :", 0)
print("Case distribution preserved:", True)
print("Label distribution preserved:", True)
print("Raw sources modified       :", False)
print("Training performed         :", False)

print("\nARTIFACTS:")
for output in OUTPUTS:
    print("-", output)

print("\nFINAL STATUS:", audit["status"])
print("=" * 76)


TEMPORAL PART2 — TIMESTAMP CANONICALIZATION
Locked rows: 348
Raw CSV files: 65
Matching raw occurrences: 831
Unique matched IDs: 348
Timestamp conflicts: 0
Raw read errors: 0

TEMPORAL COVERAGE
        case  tweet_count earliest_date_wib latest_date_wib  raw_occurrences
 lombok_2018          101        2018-08-06      2018-08-14              221
   palu_2018          105        2018-09-29      2018-10-03              244
sumatra_2025          142        2025-11-29      2025-12-04              366

TEMPORAL PART2 — FINAL AUDIT
Canonical tweet rows       : 348
Unique tweet IDs           : 348
Matching raw occurrences   : 831
Canonical timestamp missing: 0
Timestamp conflicts        : 0
Case distribution preserved: True
Label distribution preserved: True
Raw sources modified       : False
Training performed         : False

ARTIFACTS:
- C:\Penelitian Sentimen Bencana Nasional\05_Temporal\outputs\timestamp_canonicalization\temporal_labeled_348_locked.csv
- C:\Penelitian Sentimen Bencana Na